# Homework 7 — Gaussian-Process Prediction for NIST Ultrasonic Calibration

**Coverage:** Lectures 21–22<br>
**Due:** Sunday, November 1, 2026, 11:59 p.m. ET<br>
**Total:** 100 points

## Instructions

- Complete this notebook in Google Colab.
- Problem 1 is a manual mathematics problem. Show every important
  step in Markdown/LaTeX, or insert one clearly legible image of
  your handwritten derivation. Code may check arithmetic only
  after the derivation is complete.
- Problem 2 is a scaffolded scientific-computing study. Use the
  supplied random seeds and do not delete setup, helper, or check
  cells.
- Your submitted notebook must run from beginning to end in a
  fresh Colab runtime without Google Drive, absolute paths, or
  additional package installation.
- Label plots and include documented units. If a legacy dataset
  has no documented units, label the quantity as normalized or
  unit-unspecified rather than inventing units. Unless stated
  otherwise, report numerical answers to at least four
  significant digits.

## Student details

- **First name:**
- **Last name:**
- **Purdue email:**


In [ ]:
import hashlib
import io
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import cho_factor, cho_solve, solve_triangular
from scipy.optimize import minimize

sns.set_theme(style="ticks", context="notebook")
plt.rcParams["figure.dpi"] = 120
np.set_printoptions(precision=6, suppress=True)

NIST_DATA_URL = (
    "https://www.itl.nist.gov/div898/strd/nls/data/"
    "LINKS/DATA/Chwirut1.dat"
)
NIST_DATA_SHA256 = (
    "d9a055dfe5af71a8754c00f073ef00f8"
    "fed2e3fd1c6fd20cea8fd62d7cc3ed84"
)

request = urllib.request.Request(
    NIST_DATA_URL,
    headers={"User-Agent": "Purdue-ME539-course-materials/1.0"},
)
with urllib.request.urlopen(request, timeout=60) as response:
    nist_payload = response.read()

actual_sha256 = hashlib.sha256(nist_payload).hexdigest()
assert actual_sha256 == NIST_DATA_SHA256, (
    "The NIST file has changed. Stop and contact the instructor."
)

nist_array = np.loadtxt(io.BytesIO(nist_payload), skiprows=60)
chwirut = pd.DataFrame(
    {
        "ultrasonic_response": nist_array[:, 0],
        "metal_distance": nist_array[:, 1],
    }
)

x = chwirut["metal_distance"].to_numpy(float)
y = chwirut["ultrasonic_response"].to_numpy(float)

interpolation = np.isin(x, [2.25, 2.50, 2.75])
extrapolation = x >= 5.0
training = ~(interpolation | extrapolation)

y_mean = y[training].mean()
y_sd = y[training].std(ddof=0)
y_standard = (y - y_mean) / y_sd

assert chwirut.shape == (214, 2)
assert np.isfinite(chwirut.to_numpy()).all()
assert np.unique(x).size == 22
assert (training.sum(), interpolation.sum(), extrapolation.sum()) == (
    159, 26, 29
)
assert set(np.unique(x[interpolation])) == {2.25, 2.50, 2.75}
assert x[extrapolation].min() == 5.0

print("NIST Chwirut1 data loaded; fixed split masks are ready.")


## Problem 1 — Gaussian-process conditioning (25 points)

Let observations be \(x=(0,2)\) days and \(y=(1,-0.5)^T\), where
response is measured in an arbitrary response unit. Use a zero
prior mean, signal standard deviation of one response unit,
length scale of one day, and kernel
\(k(x,x')=\exp[-(x-x')^2/2]\) when the numerical values of
\(x,x'\) are expressed in days. The observation-noise variance is
\(\sigma_n^2=0.04\) response-unit squared. Predict the latent
response at \(x_*=1\) day.

1. Write the joint Gaussian distribution of \((y_1,y_2,f_*)\). **(5)**
2. Construct \(K_y,k_*\) and solve \(K_y\alpha=y\). **(6)**
3. Calculate posterior mean, latent variance, and future noisy-
   observation variance. **(6)**
4. Show that the zero-noise limit interpolates at a training point
   and has zero latent variance. **(5)**
5. State units and verify symmetry/nonnegative variance. **(3)**


> **Response:** Replace this text with your work.


## Problem 2 — NIST ultrasonic calibration (75 points)

The [NIST Chwirut1 dataset](https://www.itl.nist.gov/div898/strd/nls/data/chwirut1.shtml)
contains 214 observed ultrasonic responses at 22 metal-distance
settings. Many distance settings were measured repeatedly. NIST
does not state units for either quantity on the dataset page, so
label them as **NIST-reported units** rather than inventing units.

The setup cell downloads the official NIST ASCII file at runtime
and verifies its SHA-256 checksum; the course repository does not
redistribute the data. We will compare GP interpolation across an
interior gap with extrapolation beyond the largest training
distance. Every replicate at a withheld distance is withheld, so
an identical input cannot leak into both training and test sets.


In [ ]:
JITTER = 1e-8

def pairwise_distance(x_left, x_right):
    x_left = np.asarray(x_left, float).reshape(-1)
    x_right = np.asarray(x_right, float).reshape(-1)
    return np.abs(x_left[:, None] - x_right[None, :])

def gp_negative_log_marginal_likelihood(log_parameters, kernel,
                                        x_train, y_train):
    amplitude, length_scale, noise_sd = np.exp(log_parameters)
    K = kernel(x_train, x_train, amplitude, length_scale)
    K = K + (noise_sd**2 + JITTER) * np.eye(len(x_train))
    factor = cho_factor(K, lower=True, check_finite=True)
    alpha = cho_solve(factor, y_train)
    return (
        0.5 * y_train @ alpha
        + np.log(np.diag(factor[0])).sum()
        + 0.5 * len(x_train) * np.log(2*np.pi)
    )

def fit_gp(kernel, x_train, y_train):
    initial = np.log([1.0, 1.0, 0.15])
    bounds = np.log([[0.1, 5.0], [0.05, 15.0], [0.01, 2.0]])
    result = minimize(
        gp_negative_log_marginal_likelihood,
        initial,
        args=(kernel, x_train, y_train),
        method="L-BFGS-B",
        bounds=bounds,
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result

def gp_posterior(kernel, fitted_log_parameters,
                 x_train, y_train, x_test):
    amplitude, length_scale, noise_sd = np.exp(fitted_log_parameters)
    K = kernel(x_train, x_train, amplitude, length_scale)
    K = K + (noise_sd**2 + JITTER) * np.eye(len(x_train))
    factor = cho_factor(K, lower=True, check_finite=True)
    K_cross = kernel(x_train, x_test, amplitude, length_scale)
    alpha = cho_solve(factor, y_train)
    mean = K_cross.T @ alpha
    projected = solve_triangular(factor[0], K_cross, lower=True)
    prior_variance = np.diag(
        kernel(x_test, x_test, amplitude, length_scale)
    )
    raw_latent_variance = (
        prior_variance - np.sum(projected**2, axis=0)
    )
    if raw_latent_variance.min(initial=0.0) < -1e-8:
        raise FloatingPointError(
            "A materially negative GP variance was computed."
        )
    # Clip roundoff only after rejecting a substantive violation.
    latent_variance = np.maximum(raw_latent_variance, 0.0)
    noisy_variance = latent_variance + noise_sd**2
    return mean, latent_variance, noisy_variance


### 2.1 Audit the replicated data and split (12 points)

Verify the row count, number and range of distinct distances,
missingness, response range, and replicate count at every distance.
Make one figure showing all observations, the per-distance mean,
and a mean-plus-or-minus-one-sample-SD band or error bars. Mark the
interpolation and extrapolation holdouts. Confirm that response
standardization used the training rows only, and explain why a
random row split would leak information through replicated inputs.


In [ ]:
replicate_summary = (
    chwirut.groupby("metal_distance")["ultrasonic_response"]
    .agg(["count", "mean", "std"])
    .reset_index()
)

# YOUR CODE HERE: audit the data and construct the requested plot.


> **Response:** Report the audit and explain why the split is by distance.


### 2.2 Kernels and exact GP fitting (23 points)

Implement
\[
k_{RBF}(r)=a^2e^{-r^2/(2\ell^2)},\qquad
k_{M32}(r)=a^2(1+\sqrt3r/\ell)e^{-\sqrt3r/\ell}.
\]
where $r=|x-x'|$. Implement both kernels, then use the supplied
Cholesky posterior and negative log marginal likelihood. Optimize
once per kernel in log coordinates from
`(a, ell, noise)=(1,1,0.15)`. The supplied bounds are in standardized
response units and NIST-reported distance units. Report the fitted
parameters and objective. Check kernel symmetry, the diagonal
value $a^2$, positive semidefiniteness to numerical tolerance,
and nonnegative predictive variances.


In [ ]:
def rbf_kernel(x_left, x_right, amplitude, length_scale):
    r = pairwise_distance(x_left, x_right)
    # YOUR CODE HERE
    raise NotImplementedError


def matern32_kernel(x_left, x_right, amplitude, length_scale):
    r = pairwise_distance(x_left, x_right)
    # YOUR CODE HERE
    raise NotImplementedError


def validate_kernel(kernel):
    check_x = np.array([0.5, 1.25, 3.0, 4.75])
    amplitude = 1.7
    K = kernel(check_x, check_x, amplitude, 0.8)
    assert K.shape == (4, 4)
    assert np.allclose(K, K.T, atol=1e-12)
    assert np.allclose(np.diag(K), amplitude**2, atol=1e-12)
    assert np.linalg.eigvalsh(K).min() >= -1e-10


# Required contract after you complete this cell:
# fit_results[name] = {
#     "kernel": callable,
#     "optimization": scipy OptimizeResult,
#     "parameters": np.array([amplitude, length_scale, noise_sd]),
# }
fit_results = {}

def validate_fit_results(results):
    assert set(results) == {"RBF", "Matern-3/2"}
    for result in results.values():
        assert callable(result["kernel"])
        assert result["optimization"].success
        parameters = np.asarray(result["parameters"], float)
        assert parameters.shape == (3,)
        assert np.isfinite(parameters).all()
        assert np.all(parameters > 0)

# YOUR CODE HERE: validate both kernels, populate fit_results on
# the training split, report parameters/objectives, and call
# validate_fit_results(fit_results).


### 2.3 Posterior prediction (15 points)

For both kernels, predict on a dense distance grid and at every
held-out observation. Convert standardized-response predictions
back using
\[
\mu_y=y_{mean}+y_{sd}\mu_{std},\qquad
v_y=y_{sd}^2v_{std}.
\]
Make aligned kernel panels showing training observations, both
holdouts, posterior mean, and the 95% **noisy-observation** interval.
Use common axis limits so the panels are visually comparable.


In [ ]:
prediction_grid = np.linspace(x.min(), x.max(), 500)
prediction_inputs = {
    "grid": prediction_grid,
    "interpolation": x[interpolation],
    "extrapolation": x[extrapolation],
}

# Required contract after you complete this cell:
# predictions[kernel_name][split_name] = {
#     "mean": one-dimensional array,
#     "latent_variance": one-dimensional array,
#     "noisy_variance": one-dimensional array,
# }
predictions = {}

def validate_predictions(results):
    assert set(results) == {"RBF", "Matern-3/2"}
    for kernel_result in results.values():
        assert set(kernel_result) == set(prediction_inputs)
        for split_name, values in kernel_result.items():
            expected_shape = prediction_inputs[split_name].shape
            assert set(values) == {
                "mean", "latent_variance", "noisy_variance"
            }
            for quantity in values.values():
                quantity = np.asarray(quantity, float)
                assert quantity.shape == expected_shape
                assert np.isfinite(quantity).all()
            assert np.all(values["latent_variance"] >= 0)
            assert np.all(
                values["noisy_variance"]
                >= values["latent_variance"]
            )

# YOUR CODE HERE: populate predictions in original response units,
# call validate_predictions(predictions), and make the two panels.


### 2.4 Held-out evaluation (15 points)

For each kernel and holdout separately, report RMSE for the
NIST ultrasonic response and mean negative log predictive density
from the noisy predictive Gaussian,
\[
\frac1N\sum_i\frac12\left[\log(2\pi v_i)
+\frac{(y_i-\mu_i)^2}{v_i}\right].
\]
Also report the fraction inside the 95% noisy-observation interval.
Do not combine interpolation and extrapolation into one score.


In [ ]:
def held_out_metrics(y_true, mean, noisy_variance):
    y_true = np.asarray(y_true, float)
    mean = np.asarray(mean, float)
    noisy_variance = np.asarray(noisy_variance, float)
    assert y_true.shape == mean.shape == noisy_variance.shape
    assert np.all(noisy_variance > 0)
    # YOUR CODE HERE
    raise NotImplementedError


# Required final schema and ordering:
METRIC_COLUMNS = [
    "kernel", "holdout", "rmse", "mean_nlpd", "coverage_95"
]
metrics = pd.DataFrame(columns=METRIC_COLUMNS)

def validate_metrics(table):
    assert list(table.columns) == METRIC_COLUMNS
    assert table.shape == (4, 5)
    assert set(table["kernel"]) == {"RBF", "Matern-3/2"}
    assert set(table["holdout"]) == {
        "interpolation", "extrapolation"
    }
    numeric = table[["rmse", "mean_nlpd", "coverage_95"]]
    assert np.isfinite(numeric.to_numpy(float)).all()
    assert table["coverage_95"].between(0, 1).all()

# YOUR CODE HERE: calculate the four rows in the required order
# (RBF interpolation, RBF extrapolation, Matern-3/2 interpolation,
# Matern-3/2 extrapolation), then call validate_metrics(metrics).


### 2.5 Scientific interpretation (10 points)

Use your replicate summary to assess the constant-noise assumption.
Compare the two kernels separately for interpolation and
extrapolation, and explain why extrapolation uncertainty should not
be read as a guarantee. Then state one concrete limitation of a
stationary, homoscedastic GP for this calibration dataset and one
model or experiment change that would address it. Support your
conclusions with numerical or graphical evidence from this study.


> **Response:** Replace this text with your work.
